# Train world model

**Stop any run that shows solid-color recons** — that recipe collapsed.

This notebook uses the fixed balanced recipe:
- KL scale back to 1.0 (so `z` is not pure noise)
- Aux encoder→pixel recon (keeps perception sharp)
- L1 recon at moderate scale

Watch two recon curves: `recon` (`[h,z]`) and `recon_embed` (encoder). The
embed path should get readable structure first; `[h,z]` follows.

Kernel → Restart → run all. `RESUME = None`.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import clear_output, display
from torch.utils.tensorboard import SummaryWriter

import models.decoder as decoder_mod
import models.world_model as wm_mod
import training.losses as losses_mod
from models.preprocess import nchw_float_to_nhwc_uint8, nhwc_uint8_to_nchw_float
from training.device import get_device
from training.replay_buffer import ReplayBuffer, collect_random_episodes

importlib.reload(losses_mod)
importlib.reload(decoder_mod)
importlib.reload(wm_mod)
world_model_loss = losses_mod.world_model_loss
WorldModel = wm_mod.WorldModel

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

%matplotlib inline

CONFIG = Path("configs/m3_world_model.yaml")
RESUME = None
STEPS_OVERRIDE: int | None = None
RECOLLECT = False  # True only if you still need the 80-episode dump

with CONFIG.open() as f:
    cfg = yaml.safe_load(f)

seed = int(cfg["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = get_device()
print(f"device: {device}")
print(
    f"recon_scale={cfg['train']['recon_scale']} "
    f"recon_embed_scale={cfg['train'].get('recon_embed_scale')} "
    f"kl_scale={cfg['train']['kl_scale']} free_nats={cfg['train']['free_nats']}"
)


In [ ]:
replay_path = Path(cfg["collect"]["out_path"])
if RECOLLECT or not replay_path.exists():
    print(f"collecting {cfg['collect']['num_episodes']} episodes…")
    buf = collect_random_episodes(
        env_id=str(cfg["env"]["id"]),
        num_episodes=int(cfg["collect"]["num_episodes"]),
        max_episode_steps=int(cfg["collect"]["max_episode_steps"]),
        action_dim=int(cfg["env"]["action_dim"]),
        seed=seed,
    )
    replay_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(buf.state_dict(), replay_path)
    print(f"wrote {replay_path}: episodes={len(buf)} steps={buf.num_steps}")
else:
    print(f"using existing {replay_path}")

buffer = ReplayBuffer(seed=seed)
buffer.load_state_dict(torch.load(replay_path, weights_only=False))
print(f"replay: episodes={len(buffer)} steps={buffer.num_steps}")


In [ ]:
enc, rssm, dec, heads, train_cfg = (
    cfg["encoder"],
    cfg["rssm"],
    cfg.get("decoder", {}),
    cfg.get("heads", {}),
    cfg["train"],
)
model = WorldModel.from_config_dims(
    embed_dim=int(enc["embed_dim"]),
    encoder_channels=tuple(int(c) for c in enc["channels"]),
    action_dim=int(cfg["env"]["action_dim"]),
    deter_dim=int(rssm["deter_dim"]),
    stoch=int(rssm["stoch"]),
    classes=int(rssm["classes"]),
    hidden=int(rssm["hidden"]),
    unimix=float(rssm.get("unimix", 0.01)),
    act=str(rssm.get("act", "silu")),
    initial=str(rssm.get("initial", "learned")),
    rec_depth=int(rssm.get("rec_depth", 1)),
    decoder_channels=tuple(int(c) for c in dec.get("channels", [512, 256, 128, 64])),
    head_hidden=int(heads.get("hidden", 512)),
    head_layers=int(heads.get("layers", 2)),
).to(device)
optim = torch.optim.Adam(model.parameters(), lr=float(train_cfg["lr"]))
start_step = 0
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


In [ ]:
log_dir = Path(train_cfg["log_dir"])
ckpt_dir = Path(train_cfg["checkpoint_dir"])
results_dir = Path(train_cfg["results_dir"])
for p in (log_dir, ckpt_dir, results_dir):
    p.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(log_dir=str(log_dir))

steps = int(STEPS_OVERRIDE) if STEPS_OVERRIDE is not None else int(train_cfg["steps"])
batch_size = int(train_cfg["batch_size"])
seq_len = int(train_cfg["seq_len"])
log_every = int(train_cfg["log_every"])
image_every = int(train_cfg["image_every"])
ckpt_every = int(train_cfg["checkpoint_every"])

history: list[dict] = []
last_recon_vis = None


def show_progress(history, recon_vis):
    clear_output(wait=True)
    xs = [h["step"] for h in history]
    fig = plt.figure(figsize=(12, 7))
    ax0 = fig.add_subplot(2, 2, 1)
    for key, color in [
        ("recon", "#264653"),
        ("recon_embed", "#8ab17d"),
        ("reward", "#e76f51"),
        ("continue", "#2a9d8f"),
        ("kl", "#e9c46a"),
    ]:
        ax0.plot(xs, [h[key] for h in history], label=key, color=color)
    ax0.set_title("unscaled loss terms")
    ax0.legend(fontsize=8)

    ax1 = fig.add_subplot(2, 2, 2)
    ax1.plot(xs, [h["kl_dyn_raw"] for h in history], label="kl_dyn_raw", color="#e76f51")
    ax1.plot(xs, [h["kl_rep_raw"] for h in history], label="kl_rep_raw", color="#2a9d8f")
    ax1.axhline(float(train_cfg["free_nats"]), color="gray", ls="--", label="free_nats")
    ax1.set_title("KL raw")
    ax1.legend(fontsize=8)

    if recon_vis is not None:
        real, pred_hz, pred_emb = recon_vis
        for i in range(min(2, real.shape[0])):
            ax = fig.add_subplot(2, 4, 5 + i * 2)
            ax.imshow(np.concatenate([real[i].numpy(), pred_hz[i].numpy()], axis=1))
            ax.set_title(f"real | [h,z] #{i}", fontsize=8)
            ax.axis("off")
            ax = fig.add_subplot(2, 4, 6 + i * 2)
            ax.imshow(np.concatenate([real[i].numpy(), pred_emb[i].numpy()], axis=1))
            ax.set_title(f"real | embed #{i}", fontsize=8)
            ax.axis("off")
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    h = history[-1]
    print(
        f"step {h['step']:5d}/{start_step + steps}  "
        f"recon={h['recon']:.4f} emb={h['recon_embed']:.4f}  "
        f"kl_raw={h['kl_dyn_raw']:.3f}/{h['kl_rep_raw']:.3f}"
    )


model.train()
print(f"training {steps} steps on {device}…")
for step in range(start_step + 1, start_step + steps + 1):
    batch = buffer.sample(batch_size, seq_len)
    obs = batch["obs"].to(device)
    out = model(obs, batch["actions"].to(device))
    b, t = obs.shape[:2]
    obs_f = nhwc_uint8_to_nchw_float(obs.reshape(b * t, *obs.shape[2:])).view(b, t, 3, 64, 64)
    loss = world_model_loss(
        obs=obs_f,
        recon=out.recon,
        recon_embed=out.recon_embed,
        reward=batch["rewards"].to(device),
        reward_pred=out.reward_pred,
        cont=batch["cont"].to(device),
        cont_logit=out.cont_logit,
        post_logits=out.rssm.posterior_logits,
        prior_logits=out.rssm.prior_logits,
        unimix=model.rssm.unimix,
        dyn_scale=float(train_cfg["dyn_scale"]),
        rep_scale=float(train_cfg["rep_scale"]),
        free_nats=float(train_cfg["free_nats"]),
        recon_scale=float(train_cfg["recon_scale"]),
        recon_embed_scale=float(train_cfg.get("recon_embed_scale", 1.0)),
        reward_scale=float(train_cfg["reward_scale"]),
        continue_scale=float(train_cfg["continue_scale"]),
        kl_scale=float(train_cfg["kl_scale"]),
        recon_loss_type=str(train_cfg.get("recon_loss", "l1")),
    )
    optim.zero_grad(set_to_none=True)
    loss.total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 100.0)
    optim.step()

    metrics = {
        "step": step,
        "total": float(loss.total.detach()),
        "recon": float(loss.recon.detach()),
        "recon_embed": float(loss.recon_embed.detach()),
        "reward": float(loss.reward.detach()),
        "continue": float(loss.continue_loss.detach()),
        "kl": float(loss.kl.detach()),
        "kl_dyn": float(loss.kl_dyn.detach()),
        "kl_rep": float(loss.kl_rep.detach()),
        "kl_dyn_raw": float(loss.kl_dyn_raw.detach()),
        "kl_rep_raw": float(loss.kl_rep_raw.detach()),
    }

    if step % log_every == 0 or step == start_step + 1:
        for k, v in metrics.items():
            if k != "step":
                writer.add_scalar(f"m3/{k}", v, step)
        history.append(metrics)
        show_progress(history, last_recon_vis)

    if step % image_every == 0 or step == start_step + 1:
        model.eval()
        with torch.no_grad():
            vis = buffer.sample(min(4, batch_size), seq_len)
            v_out = model(vis["obs"].to(device), vis["actions"].to(device))
            real = vis["obs"][:, 0].cpu()
            pred_hz = nchw_float_to_nhwc_uint8(v_out.recon[:, 0].detach().cpu())
            pred_emb = nchw_float_to_nhwc_uint8(v_out.recon_embed[:, 0].detach().cpu())
            last_recon_vis = (real, pred_hz, pred_emb)
            from PIL import Image

            strips = [
                np.concatenate([real[i].numpy(), pred_hz[i].numpy(), pred_emb[i].numpy()], axis=1)
                for i in range(real.shape[0])
            ]
            Image.fromarray(np.concatenate(strips, axis=0), mode="RGB").save(
                results_dir / f"recon_step_{step:05d}.png"
            )
        model.train()
        if history:
            show_progress(history, last_recon_vis)

    if step % ckpt_every == 0:
        path = ckpt_dir / f"ckpt_step_{step:05d}.pt"
        torch.save({"step": step, "model": model.state_dict(), "optim": optim.state_dict()}, path)
        print(f"wrote {path}")

final = ckpt_dir / "ckpt_final.pt"
torch.save({"step": start_step + steps, "model": model.state_dict(), "optim": optim.state_dict()}, final)
(results_dir / "train_metrics.json").write_text(json.dumps(history, indent=2))
writer.flush(); writer.close()
show_progress(history, last_recon_vis)
print("done", final)
